In [1]:
import numpy as np
import pandas as pd

# **Dataset Preparation**

In [2]:
df = pd.read_csv('/content/questions.csv')

In [3]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [4]:
df.dropna(inplace=True)

In [5]:
df.isnull().sum()

,0
id,0
qid1,0
qid2,0
question1,0
question2,0
is_duplicate,0


In [6]:
df.drop(columns=['id','qid1', 'qid2'], inplace=True)
df.head()

,question1,question2,is_duplicate
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


# **Data Cleaning**

In [7]:
import re
from bs4 import BeautifulSoup

import warnings
warnings.filterwarnings('ignore')

In [8]:
import re
def preprocess(q):

    q = str(q).lower().strip()

    # Replace certain special characters with their string equivalents
    q = q.replace('%', ' percent')
    q = q.replace('$', ' dollar ')
    q = q.replace('₹', ' rupee ')
    q = q.replace('€', ' euro ')
    q = q.replace('@', ' at ')

    # The pattern '[math]' appears around 900 times in the whole dataset.
    q = q.replace('[math]', '')

    # Replacing some numbers with string equivalents (not perfect, can be done better to account for more cases)
    q = q.replace(',000,000,000 ', 'b ')
    q = q.replace(',000,000 ', 'm ')
    q = q.replace(',000 ', 'k ')
    q = re.sub(r'([0-9]+)000000000', r'\1b', q)
    q = re.sub(r'([0-9]+)000000', r'\1m', q)
    q = re.sub(r'([0-9]+)000', r'\1k', q)

    # Decontracting words
    # https://en.wikipedia.org/wiki/Wikipedia%3aList_of_English_contractions
    # https://stackoverflow.com/a/19794953
    contractions = {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "can not",
    "can't've": "can not have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how'll": "how will",
    "how's": "how is",
    "i'd": "i would",
    "i'd've": "i would have",
    "i'll": "i will",
    "i'll've": "i will have",
    "i'm": "i am",
    "i've": "i have",
    "isn't": "is not",
    "it'd": "it would",
    "it'd've": "it would have",
    "it'll": "it will",
    "it'll've": "it will have",
    "it's": "it is",
    "let's": "let us",
    "ma'am": "madam",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "o'clock": "of the clock",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shan't've": "shall not have",
    "she'd": "she would",
    "she'd've": "she would have",
    "she'll": "she will",
    "she'll've": "she will have",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so as",
    "that'd": "that would",
    "that'd've": "that would have",
    "that's": "that is",
    "there'd": "there would",
    "there'd've": "there would have",
    "there's": "there is",
    "they'd": "they would",
    "they'd've": "they would have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they're": "they are",
    "they've": "they have",
    "to've": "to have",
    "wasn't": "was not",
    "we'd": "we would",
    "we'd've": "we would have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what'll've": "what will have",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "when's": "when is",
    "when've": "when have",
    "where'd": "where did",
    "where's": "where is",
    "where've": "where have",
    "who'll": "who will",
    "who'll've": "who will have",
    "who's": "who is",
    "who've": "who have",
    "why's": "why is",
    "why've": "why have",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "you'd": "you would",
    "you'd've": "you would have",
    "you'll": "you will",
    "you'll've": "you will have",
    "you're": "you are",
    "you've": "you have"
    }

    q_decontracted = []

    for word in q.split():
        if word in contractions:
            word = contractions[word]

        q_decontracted.append(word)

    q = ' '.join(q_decontracted)
    q = q.replace("'ve", " have")
    q = q.replace("n't", " not")
    q = q.replace("'re", " are")
    q = q.replace("'ll", " will")

    # Removing HTML tags
    q = BeautifulSoup(q)
    q = q.get_text()

    # Remove punctuations
    pattern = re.compile('\W')
    q = re.sub(pattern, ' ', q).strip()


    return q


In [9]:
df['q1_clean'] = df['question1'].apply(preprocess)
df['q2_clean'] = df['question2'].apply(preprocess)

# **Tokenization**

In [10]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')

df['q1_token'] = df['q1_clean'].apply(word_tokenize)
df['q2_token'] = df['q2_clean'].apply(word_tokenize)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [11]:
df.head()

,question1,question2,is_duplicate,q1_clean,q2_clean,q1_token,q2_token
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,"[what, is, the, step, by, step, guide, to, inv...","[what, is, the, step, by, step, guide, to, inv..."
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0,what is the story of kohinoor koh i noor dia...,what would happen if the indian government sto...,"[what, is, the, story, of, kohinoor, koh, i, n...","[what, would, happen, if, the, indian, governm..."
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,"[how, can, i, increase, the, speed, of, my, in...","[how, can, internet, speed, be, increased, by,..."
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0,why am i mentally very lonely how can i solve it,find the remainder when 23 24 math is divi...,"[why, am, i, mentally, very, lonely, how, can,...","[find, the, remainder, when, 23, 24, math, is,..."
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0,which one dissolve in water quikly sugar salt...,which fish would survive in salt water,"[which, one, dissolve, in, water, quikly, suga...","[which, fish, would, survive, in, salt, water]"


In [12]:
all_word = []
for word in df['q1_token'].tolist() + df['q2_token'].tolist():
  all_word.extend(word)

vocab = {'<UNK>' : 0}
for word in all_word:
  if word not in vocab:
    vocab[word] = len(vocab)

In [13]:
len(vocab)

86370

In [14]:
def tokenize_sentence(row):
  tokenized_sentence = []
  for word in row:
    if word in vocab:
      tokenized_sentence.append(vocab[word])
    else:
      tokenized_sentence.append(vocab['<UNK>'])
  return tokenized_sentence


In [15]:
df['q1_tokenized'] = df['q1_token'].apply(tokenize_sentence)
df['q2_tokenized'] = df['q2_token'].apply(tokenize_sentence)

In [16]:
df.head()

,question1,question2,is_duplicate,q1_clean,q2_clean,q1_token,q2_token,q1_tokenized,q2_tokenized
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,"[what, is, the, step, by, step, guide, to, inv...","[what, is, the, step, by, step, guide, to, inv...","[1, 2, 3, 4, 5, 4, 6, 7, 8, 9, 10, 11, 9, 12]","[1, 2, 3, 4, 5, 4, 6, 7, 8, 9, 10, 11]"
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0,what is the story of kohinoor koh i noor dia...,what would happen if the indian government sto...,"[what, is, the, story, of, kohinoor, koh, i, n...","[what, would, happen, if, the, indian, governm...","[1, 2, 3, 13, 14, 15, 16, 17, 18, 19]","[1, 113, 863, 316, 3, 441, 205, 4612, 3, 15, 1..."
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,"[how, can, i, increase, the, speed, of, my, in...","[how, can, internet, speed, be, increased, by,...","[20, 21, 17, 22, 3, 23, 14, 24, 25, 26, 27, 28...","[20, 21, 25, 23, 64, 1833, 5, 564, 168, 40693]"
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0,why am i mentally very lonely how can i solve it,find the remainder when 23 24 math is divi...,"[why, am, i, mentally, very, lonely, how, can,...","[find, the, remainder, when, 23, 24, math, is,...","[31, 32, 17, 33, 34, 35, 20, 21, 17, 36, 37]","[81, 3, 10876, 67, 3045, 8503, 1425, 2, 12651,..."
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0,which one dissolve in water quikly sugar salt...,which fish would survive in salt water,"[which, one, dissolve, in, water, quikly, suga...","[which, fish, would, survive, in, salt, water]","[38, 39, 40, 9, 41, 42, 43, 44, 45, 46, 47, 48...","[38, 3206, 113, 5747, 9, 44, 41]"


In [17]:
q1_maxlen = df['q1_tokenized'].apply(len).tolist()
q2_maxlen = df['q2_tokenized'].apply(len).tolist()

print(max(q1_maxlen))
print(max(q2_maxlen))

128
249


In [18]:
MAX_LEN_Q1 = 128
MAX_LEN_Q2 = 249

def pad_or_truncate(ids, max_len):
    if len(ids) >= max_len:
        return ids[:max_len]                        # truncate
    else:
        return ids + [0] * (max_len - len(ids))     # pad with 0

df['q1_padded'] = df['q1_tokenized'].apply(lambda x: pad_or_truncate(x, MAX_LEN_Q1))
df['q2_padded'] = df['q2_tokenized'].apply(lambda x: pad_or_truncate(x, MAX_LEN_Q2))

In [19]:
df.head()

,question1,question2,is_duplicate,q1_clean,q2_clean,q1_token,q2_token,q1_tokenized,q2_tokenized,q1_padded,q2_padded
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,"[what, is, the, step, by, step, guide, to, inv...","[what, is, the, step, by, step, guide, to, inv...","[1, 2, 3, 4, 5, 4, 6, 7, 8, 9, 10, 11, 9, 12]","[1, 2, 3, 4, 5, 4, 6, 7, 8, 9, 10, 11]","[1, 2, 3, 4, 5, 4, 6, 7, 8, 9, 10, 11, 9, 12, ...","[1, 2, 3, 4, 5, 4, 6, 7, 8, 9, 10, 11, 0, 0, 0..."
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0,what is the story of kohinoor koh i noor dia...,what would happen if the indian government sto...,"[what, is, the, story, of, kohinoor, koh, i, n...","[what, would, happen, if, the, indian, governm...","[1, 2, 3, 13, 14, 15, 16, 17, 18, 19]","[1, 113, 863, 316, 3, 441, 205, 4612, 3, 15, 1...","[1, 2, 3, 13, 14, 15, 16, 17, 18, 19, 0, 0, 0,...","[1, 113, 863, 316, 3, 441, 205, 4612, 3, 15, 1..."
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,"[how, can, i, increase, the, speed, of, my, in...","[how, can, internet, speed, be, increased, by,...","[20, 21, 17, 22, 3, 23, 14, 24, 25, 26, 27, 28...","[20, 21, 25, 23, 64, 1833, 5, 564, 168, 40693]","[20, 21, 17, 22, 3, 23, 14, 24, 25, 26, 27, 28...","[20, 21, 25, 23, 64, 1833, 5, 564, 168, 40693,..."
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0,why am i mentally very lonely how can i solve it,find the remainder when 23 24 math is divi...,"[why, am, i, mentally, very, lonely, how, can,...","[find, the, remainder, when, 23, 24, math, is,...","[31, 32, 17, 33, 34, 35, 20, 21, 17, 36, 37]","[81, 3, 10876, 67, 3045, 8503, 1425, 2, 12651,...","[31, 32, 17, 33, 34, 35, 20, 21, 17, 36, 37, 0...","[81, 3, 10876, 67, 3045, 8503, 1425, 2, 12651,..."
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0,which one dissolve in water quikly sugar salt...,which fish would survive in salt water,"[which, one, dissolve, in, water, quikly, suga...","[which, fish, would, survive, in, salt, water]","[38, 39, 40, 9, 41, 42, 43, 44, 45, 46, 47, 48...","[38, 3206, 113, 5747, 9, 44, 41]","[38, 39, 40, 9, 41, 42, 43, 44, 45, 46, 47, 48...","[38, 3206, 113, 5747, 9, 44, 41, 0, 0, 0, 0, 0..."


In [20]:
df[df['q1_tokenized'].apply(len) == 128]

,question1,question2,is_duplicate,q1_clean,q2_clean,q1_token,q2_token,q1_tokenized,q2_tokenized,q1_padded,q2_padded
35108,"Like everyone else (here in U.S), I work with ...",When is it ok to force people to do things aga...,0,like everyone else here in u s i work with ...,when is it ok to force people to do things aga...,"[like, everyone, else, here, in, u, s, i, work...","[when, is, it, ok, to, force, people, to, do, ...","[98, 2037, 394, 2886, 9, 4101, 121, 17, 517, 1...","[67, 2, 37, 2865, 7, 1748, 218, 7, 68, 868, 63...","[98, 2037, 394, 2886, 9, 4101, 121, 17, 517, 1...","[67, 2, 37, 2865, 7, 1748, 218, 7, 68, 868, 63..."


In [21]:
df[df['q2_tokenized'].apply(len) == 249]

,question1,question2,is_duplicate,q1_clean,q2_clean,q1_token,q2_token,q1_tokenized,q2_tokenized,q1_padded,q2_padded
18057,Im moving to NY. My Dr gave me 2 refills of Xa...,Heartbreak? Heartbreak? She's my girlfriend fo...,0,im moving to ny my dr gave me 2 refills of xa...,heartbreak heartbreak she is my girlfriend f...,"[im, moving, to, ny, my, dr, gave, me, 2, refi...","[heartbreak, heartbreak, she, is, my, girlfrie...","[1456, 2971, 7, 15970, 24, 3932, 2810, 60, 727...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037...","[1456, 2971, 7, 15970, 24, 3932, 2810, 60, 727...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037..."
51957,Most of us are never going to be famous. I'm m...,Heartbreak? Heartbreak? She's my girlfriend fo...,0,most of us are never going to be famous i am ...,heartbreak heartbreak she is my girlfriend f...,"[most, of, us, are, never, going, to, be, famo...","[heartbreak, heartbreak, she, is, my, girlfrie...","[199, 14, 108, 99, 2048, 425, 7, 64, 2548, 17,...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037...","[199, 14, 108, 99, 2048, 425, 7, 64, 2548, 17,...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037..."
75742,I'm in 12 steps program. Been sober for 4 year...,Heartbreak? Heartbreak? She's my girlfriend fo...,0,i am in 12 steps program been sober for 4 yea...,heartbreak heartbreak she is my girlfriend f...,"[i, am, in, 12, steps, program, been, sober, f...","[heartbreak, heartbreak, she, is, my, girlfrie...","[17, 32, 9, 1774, 2089, 1606, 1741, 5856, 117,...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037...","[17, 32, 9, 1774, 2089, 1606, 1741, 5856, 117,...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037..."
94492,"What did you think of Obama yelling at ""Chatty...",Heartbreak? Heartbreak? She's my girlfriend fo...,0,what did you think of obama yelling at chatty...,heartbreak heartbreak she is my girlfriend f...,"[what, did, you, think, of, obama, yelling, at...","[heartbreak, heartbreak, she, is, my, girlfrie...","[1, 241, 69, 219, 14, 220, 15965, 161, 35532, ...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037...","[1, 241, 69, 219, 14, 220, 15965, 161, 35532, ...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037..."
118598,Am I going to be forever alone? I meant I have...,Heartbreak? Heartbreak? She's my girlfriend fo...,0,am i going to be forever alone i meant i have...,heartbreak heartbreak she is my girlfriend f...,"[am, i, going, to, be, forever, alone, i, mean...","[heartbreak, heartbreak, she, is, my, girlfrie...","[32, 17, 425, 7, 64, 5169, 6569, 17, 3441, 17,...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037...","[32, 17, 425, 7, 64, 5169, 6569, 17, 3441, 17,...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037..."
130799,I love singing. It brings me joy. But my voice...,Heartbreak? Heartbreak? She's my girlfriend fo...,0,i love singing it brings me joy but my voice...,heartbreak heartbreak she is my girlfriend f...,"[i, love, singing, it, brings, me, joy, but, m...","[heartbreak, heartbreak, she, is, my, girlfrie...","[17, 565, 4966, 37, 7973, 60, 7974, 1168, 24, ...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037...","[17, 565, 4966, 37, 7973, 60, 7974, 1168, 24, ...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037..."
131671,Is it possible that I am having doubts about m...,Heartbreak? Heartbreak? She's my girlfriend fo...,0,is it possible that i am having doubts about m...,heartbreak heartbreak she is my girlfriend f...,"[is, it, possible, that, i, am, having, doubts...","[heartbreak, heartbreak, she, is, my, girlfrie...","[2, 37, 734, 57, 17, 32, 2256, 20802, 59, 24, ...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037...","[2, 37, 734, 57, 17, 32, 2256, 20802, 59, 24, ...","[13585, 13585, 318, 2, 24, 237, 117, 652, 1037..."
166739,My boyfriend told me that he likes another gir...,Heartbreak? Heartbreak? She's my girlfriend fo...,0,my boyfriend told me that he likes another gir...,heartbreak heartbreak she is my girlfriend f...,"[my, boyfriend, told, me, that, he, likes, ano...",

In [22]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class QuoraDataset(Dataset):
    def __init__(self, df):
        self.q1 = torch.tensor(np.array(df['q1_padded'].tolist()), dtype=torch.long)
        self.q2 = torch.tensor(np.array(df['q2_padded'].tolist()), dtype=torch.long)
        self.labels = torch.tensor(df['is_duplicate'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.q1[idx], self.q2[idx], self.labels[idx]

# Train/test split
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = QuoraDataset(train_df.reset_index(drop=True))
test_dataset  = QuoraDataset(test_df.reset_index(drop=True))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

In [23]:
import torch.nn as nn

class QuoraLSTM(nn.Module):
  def __init__(self, vocab_size, hidden_size, embed_dim, num_layers, dropout = 0.3):
    super(QuoraLSTM, self).__init__()
    self.embedding = nn.Embedding(
        vocab_size,
        embed_dim
        )

    self.bilstm = nn.LSTM(
        bidirectional = True,
        batch_first = True,
        input_size = embed_dim,
        hidden_size = hidden_size,
        num_layers = num_layers,
        dropout = dropout if num_layers>1 else 0
        )

    self.dropout = nn.Dropout(dropout)

    self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 4, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 1),
            nn.Sigmoid()
        )

  def encode(self, x):
      # x: (batch, seq_len)
      emb = self.dropout(self.embedding(x))      # (batch, seq_len, embed_dim)
      _, (hn, _) = self.bilstm(emb)
      # hn: (num_layers*2, batch, hidden_size)
      # hn[-2] = last layer forward
      # hn[-1] = last layer backward
      out = torch.cat((hn[-2], hn[-1]), dim=1)   # (batch, hidden_size*2)
      return out

  def forward(self, q1, q2):
      q1_enc = self.encode(q1)                            # (batch, hidden*2)
      q2_enc = self.encode(q2)                            # (batch, hidden*2)
      combined = torch.cat((q1_enc, q2_enc), dim=1)       # (batch, hidden*4)
      out = self.classifier(combined)                     # (batch, 1)
      return out.squeeze(1)                               # (batch,)

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = QuoraLSTM(
    vocab_size = len(vocab),
    hidden_size = 128,
    embed_dim = 128,
    num_layers = 2,
    dropout = 0.3
).to(device)

print(model)

QuoraLSTM(
  (embedding): Embedding(86370, 128)
  (bilstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
    (13): Sigmoid()
  )
)


In [25]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

EPOCHS    = 40
best_loss = float('inf')

for epoch in range(EPOCHS):

    # ── Train ──────────────────────────────────────────
    model.train()
    train_loss  = 0
    train_correct = 0
    train_total   = 0

    for q1, q2, labels in train_loader:
        q1, q2, labels = q1.to(device), q2.to(device), labels.to(device)

        optimizer.zero_grad()
        preds = model(q1, q2)                        # forward pass
        loss  = criterion(preds, labels)
        loss.backward()                              # backprop
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()

        train_loss += loss.item()
        predicted    = (preds >= 0.5).float()
        train_correct += (predicted == labels).sum().item()
        train_total   += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    train_acc      = train_correct / train_total

    # ── Validation ─────────────────────────────────────
    model.eval()
    val_loss    = 0
    val_correct = 0
    val_total   = 0

    with torch.no_grad():
        for q1, q2, labels in test_loader:
            q1, q2, labels = q1.to(device), q2.to(device), labels.to(device)
            preds = model(q1, q2)
            loss  = criterion(preds, labels)

            val_loss    += loss.item()
            predicted    = (preds >= 0.5).float()
            val_correct += (predicted == labels).sum().item()
            val_total   += labels.size(0)

    avg_val_loss = val_loss / len(test_loader)
    val_acc      = val_correct / val_total

    # LR scheduler step
    scheduler.step(avg_val_loss)

    # Save best model
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model.state_dict(), "best_bilstm.pt")
        print(f"  ✅ Best model saved!")

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"| Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} "
          f"| Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} "
          f"| LR: {optimizer.param_groups[0]['lr']:.6f}")

  ✅ Best model saved!
Epoch [1/30] | Train Loss: 0.5424 | Train Acc: 0.7226 | Val Loss: 0.4840 | Val Acc: 0.7637 | LR: 0.001000
  ✅ Best model saved!
Epoch [2/30] | Train Loss: 0.4902 | Train Acc: 0.7614 | Val Loss: 0.4627 | Val Acc: 0.7803 | LR: 0.001000
  ✅ Best model saved!
Epoch [3/30] | Train Loss: 0.4670 | Train Acc: 0.7767 | Val Loss: 0.4395 | Val Acc: 0.7932 | LR: 0.001000
  ✅ Best model saved!
Epoch [4/30] | Train Loss: 0.4509 | Train Acc: 0.7868 | Val Loss: 0.4314 | Val Acc: 0.7958 | LR: 0.001000
Epoch [5/30] | Train Loss: 0.4399 | Train Acc: 0.7938 | Val Loss: 0.4320 | Val Acc: 0.7956 | LR: 0.001000
  ✅ Best model saved!
Epoch [6/30] | Train Loss: 0.4341 | Train Acc: 0.7975 | Val Loss: 0.4185 | Val Acc: 0.8045 | LR: 0.001000
  ✅ Best model saved!
Epoch [7/30] | Train Loss: 0.4282 | Train Acc: 0.8009 | Val Loss: 0.4144 | Val Acc: 0.8082 | LR: 0.001000
  ✅ Best model saved!
Epoch [8/30] | Train Loss: 0.4243 | Train Acc: 0.8028 | Val Loss: 0.4103 | Val Acc: 0.8105 | LR: 0.00100

In [25]:
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
import matplotlib.pyplot as plt
import seaborn as sns

# Load best model
model.load_state_dict(torch.load("best_bilstm.pt"))
model.eval()

all_preds  = []
all_probs  = []
all_labels = []

with torch.no_grad():
    for q1, q2, labels in test_loader:
        q1, q2 = q1.to(device), q2.to(device)
        probs   = model(q1, q2).cpu()
        preds   = (probs >= 0.5).float()

        all_probs.extend(probs.numpy())
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

# ── Metrics ────────────────────────────────────────────
print("=" * 50)
print(f"  Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(all_labels, all_probs):.4f}")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(
    all_labels, all_preds,
    target_names=['Not Duplicate', 'Duplicate']
))

# ── Confusion Matrix ────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Duplicate', 'Duplicate'],
            yticklabels=['Not Duplicate', 'Duplicate'])
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

In [ ]:
# ── SAVE ALL ───────────────────────────────────────────
import pickle

torch.save(model.state_dict(), "bilstm_quora.pt")

with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

config = {
    'vocab_size'  : VOCAB_SIZE,
    'embed_dim'   : EMBED_DIM,
    'hidden_size' : HIDDEN_SIZE,
    'num_layers'  : NUM_LAYERS,
    'dropout'     : DROPOUT,
    'max_len_q1'  : MAX_LEN_Q1,
    'max_len_q2'  : MAX_LEN_Q2,
}

with open("config.pkl", "wb") as f:
    pickle.dump(config, f)

print("✅ Model + Vocab + Config saved!")


# ── LOAD ALL ───────────────────────────────────────────
with open("vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

with open("config.pkl", "rb") as f:
    config = pickle.load(f)

model = BiLSTMQuora(
    vocab_size   = config['vocab_size'],
    embed_dim    = config['embed_dim'],
    hidden_size  = config['hidden_size'],
    num_layers   = config['num_layers'],
    dropout      = config['dropout']
).to(device)

model.load_state_dict(torch.load("bilstm_quora.pt", map_location=device))
model.eval()

MAX_LEN_Q1 = config['max_len_q1']
MAX_LEN_Q2 = config['max_len_q2']

print("✅ Model + Vocab + Config loaded!")